# Test split — scoring the fourteen rows

Steps 6 to 9 of `docs/preregistration_test.md`: the local metrics, both raters, the ten
confirmatory tests under Holm, and everything the family excludes. It reads the generation
manifest written by `notebooks/test_generation_colab.ipynb` and scores exactly the bytes that
manifest records.

Sections 1 to 3 spend nothing and run with no rater key in the kernel. Section 4 is the only
paid part, and it is the larger of the two test-pass bills.

---
## 1 — Host and working tree

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

In [ ]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

In [ ]:
!git log --oneline -1 -- docs/preregistration_test.md

In [ ]:
%pip install -r requirements.txt

In [ ]:
import getpass
import hashlib
import json
import logging
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone

import yaml

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
SPLIT = 'test'
OUT = Path('outputs')
RESULTS = Path('results')

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
# Sections 1-3 are free, so they run with no rater key in the kernel; section 4 sets them.
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), f'{var} is set; sections 1-3 make no paid call'
logging.getLogger('httpx').setLevel(logging.WARNING)
print(f'{datetime.now(timezone.utc):%Y-%m-%d %H:%M}Z, no rater key present')

---
## 2 — What is being scored

Every metric below binds to `output_sha256` in `outputs/test_manifest.json`. A row whose file
has moved since it was generated is not scored against a stale digest; it fails here.

In [ ]:
# The fourteen rows in pre-registration order. The last two are external diagnostics.
STUDY = ['zeroshot', 'random_fewshot', 'knn_fewshot', 'sparse_knn', 'afsp_margin', 'afsp_full',
         'peft', 'peft_knn', 'peft_afsp', 'rlsf_w3_0.0', 'rlsf_w3_2.0', 'rlsf_w3_6.0']
EXTERNAL = ['commercial_haiku', 'gpt56_sparse_knn']
CONDS = STUDY + EXTERNAL

N_BOOT, ALPHA, SEED = 10000, 0.05, 42
CENTROID_FINGERPRINT = 'fd5aec8d69454b02'
assert len(CONDS) == 14, CONDS
print(f'{len(CONDS)} rows, {N_BOOT} resamples, alpha {ALPHA}, seed {SEED}')

In [ ]:
MANIFEST = json.loads((OUT / 'test_manifest.json').read_text(encoding='utf-8'))
EVAL_FILE = Path(MANIFEST['eval_file']['path'])
TEST_SHA = hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest()

HASHES = json.loads(Path('data/splits/hashes.json').read_text(encoding='utf-8'))
assert TEST_SHA == HASHES['hashes']['test.jsonl'] == MANIFEST['eval_file']['sha256']
assert set(MANIFEST['rows']) == set(CONDS), sorted(set(MANIFEST['rows']) ^ set(CONDS))

SEGMENTS = [json.loads(x) for x in EVAL_FILE.open(encoding='utf-8') if x.strip()]
TEST_SRC = [r['input'] for r in SEGMENTS]
assert len(SEGMENTS) == MANIFEST['eval_file']['n'] == 1322, len(SEGMENTS)

HEAD = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print(f"generated at {MANIFEST['commit'][:12]}, scored at {HEAD[:12]}"
      f"{'' if HEAD == MANIFEST['commit'] else '  (the tree moved between the two passes)'}")
print(f"{len(SEGMENTS)} segments  {TEST_SHA[:16]}  spend so far "
      f"${MANIFEST.get('spend', {}).get('actual_usd', 0.0):.2f}")

In [ ]:
ROWS = {}
for cond in CONDS:
    path = OUT / f'{cond}_{SPLIT}.jsonl'
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == MANIFEST['rows'][cond]['output_sha256'], (
        f'{cond}: {path} is not the file the manifest records; it was regenerated after '
        f'the generation pass and the manifest no longer describes it'
    )
    ROWS[cond] = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(ROWS[cond]) == len(SEGMENTS), f'{cond}: {len(ROWS[cond])} rows'
    assert [r['input'] for r in ROWS[cond]] == TEST_SRC, f'{cond}: source order differs'
    blank = sum(1 for r in ROWS[cond] if not r['prediction'].strip())
    assert blank == 0, f'{cond}: {blank} blank predictions break the paired estimators'
    print(f'{cond:<18} {len(ROWS[cond])} rows, aligned, 0 blank  {digest[:12]}')

---
## 3 — Step 6: local scoring

Free. COMET, chrF, BLEU, the stylometric ladder, the held-out decomposition, and every
bootstrap the addendum names.

In [ ]:
!{PY} manage.py eval --conditions {' '.join(CONDS)} --split {SPLIT}

In [ ]:
from src.eval.quick import score

SURFACE = {c: score(c, OUT, SPLIT) for c in CONDS}
print(f"{'condition':<18} {'chrF':>8} {'BLEU':>8} {'markers/seg':>12}")
for cond in CONDS:
    s = SURFACE[cond]
    print(f"{cond:<18} {s['chrF']:8.2f} {s['BLEU']:8.2f} {s['marker_rate']:12.2f}")
print(f"gold test targets carry {SURFACE['zeroshot']['ref_marker_rate']:.2f} markers per segment")

In [ ]:
# COMET gets its own interpreter: requirements-comet.txt pins transformers and numpy below
# what the rest of the stack runs on.
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)
subprocess.run([COMET_PY, '-c', 'import comet; print("comet ok")'], check=True)

In [ ]:
COMET_PATH = RESULTS / f'comet_{SPLIT}.json'
subprocess.run([COMET_PY, 'manage.py', 'comet', '--conditions', *CONDS, '--split', SPLIT,
                '--results_path', str(COMET_PATH), '--batch_size', '16'], check=True)

In [ ]:
COMET = json.loads(COMET_PATH.read_text(encoding='utf-8'))
assert set(COMET) == set(CONDS), sorted(set(CONDS) - set(COMET))

ref = COMET['zeroshot']
for cond in CONDS:
    rec = COMET[cond]
    assert rec['n'] == len(SEGMENTS), (cond, rec['n'])
    assert rec['model'] == ref['model'], (cond, rec['model'])
    assert rec['sources'] == ref['sources'], f'{cond}: segments are not paired'
    assert rec['output_sha256'] == MANIFEST['rows'][cond]['output_sha256'], f'{cond}: unbound'
print(f"{ref['model']} scored all {len(CONDS)} rows on the same {len(SEGMENTS)} segments")
for cond in CONDS:
    print(f"{cond:<18} COMET {COMET[cond]['system']:.4f}")

In [ ]:
!{PY} manage.py stylometrics --conditions {' '.join(CONDS)} --split {SPLIT} --targets-split test

In [ ]:
LADDER_PATH = RESULTS / f'stylometrics_ci_ladder_{SPLIT}.json'
subprocess.run([PY, 'manage.py', 'stylometrics_ci', '--split', SPLIT, '--conditions', *CONDS,
                '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                '--results_path', str(LADDER_PATH)], check=True)

In [ ]:
STYLO = json.loads(LADDER_PATH.read_text(encoding='utf-8'))
boot = STYLO['bootstrap']
assert (boot['n_resamples'], boot['seed'], boot['alpha']) == (N_BOOT, SEED, ALPHA), boot
assert boot['n_segments'] == len(SEGMENTS), boot
assert set(STYLO['cells']) == set(CONDS), sorted(set(CONDS) - set(STYLO['cells']))
# The training-side centroid the val ladder used. A rebuilt centroid would silently rescale
# every distance in the confirmatory family.
assert STYLO['centroid']['fingerprint'] == CENTROID_FINGERPRINT, STYLO['centroid']
print(f"centroid {STYLO['centroid']['fingerprint']}, {len(STYLO['paired_all'])} pairwise intervals")
for cond in STYLO['ranking']:
    print(f"{cond:<18} stylo_dist {STYLO['cells'][cond]['stylo_dist']:.4f}  "
          f"rank {STYLO['cells'][cond]['rank']}")

In [ ]:
# Reads results/comet_test.json, so it runs after COMET. Its default figure path is the val
# figure and is redirected rather than overwritten.
subprocess.run([PY, 'manage.py', 'heldout_decomp', '--split', SPLIT,
                '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                '--figure_path', f'docs/figures/heldout_decomp_by_omega_{SPLIT}.png'], check=True)

In [ ]:
CONF_BOOT_PATH = RESULTS / f'bootstrap_comet_confirmatory_{SPLIT}.json'
subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', 'comet', '--split', SPLIT,
                '--conditions', 'knn_fewshot', 'sparse_knn', 'afsp_full', 'peft', 'peft_afsp',
                'rlsf_w3_2.0',
                '--pairs', 'peft:knn_fewshot', 'afsp_full:knn_fewshot', 'sparse_knn:knn_fewshot',
                'peft_afsp:peft', 'rlsf_w3_2.0:peft',
                '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                '--out', str(CONF_BOOT_PATH)], check=True)

In [ ]:
# The exploratory adequacy ladders. Bare --out claims results/bootstrap_<metric>_test.json,
# which is why the five confirmatory COMET intervals were written to their own path above.
for metric in ('chrf', 'bleu', 'comet'):
    subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', metric, '--split', SPLIT,
                    '--conditions', *CONDS, '--baseline', 'zeroshot', '--adjacent',
                    '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                    '--out'], check=True)

In [ ]:
LOCAL_ARTIFACTS = [COMET_PATH, LADDER_PATH, CONF_BOOT_PATH,
                   RESULTS / f'heldout_decomp_{SPLIT}.json',
                   *(RESULTS / f'bootstrap_{m}_{SPLIT}.json' for m in ('chrf', 'bleu', 'comet'))]
for p in LOCAL_ARTIFACTS:
    assert p.exists(), p
    print(f'{str(p):<52} {p.stat().st_size / 1024:8.1f} KiB')

for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), var
print('\nstep 6 complete: 0 paid calls, $0.00')

---
## 4 — Step 7: the two raters

The paid part. Both raters score all fourteen rows on the frozen rubric, each after a
twenty-segment pilot that reprices the pass before it is committed to.

In [ ]:
JUDGE_CFG_A = 'configs/judge_eval.yaml'
JUDGE_CFG_B = 'configs/judge_eval_gpt.yaml'
TAG_B = 'gpt'
PILOT_N = 20

CFG_A = yaml.safe_load(Path(JUDGE_CFG_A).read_text(encoding='utf-8'))
CFG_B = yaml.safe_load(Path(JUDGE_CFG_B).read_text(encoding='utf-8'))

assert CFG_A['judge']['model'] == 'claude-haiku-4-5', CFG_A['judge']
assert CFG_A['judge']['thinking'] is False and CFG_A['judge']['temperature'] == 0.0
assert CFG_B['judge']['model'] == 'gpt-5.6-terra', CFG_B['judge']
assert CFG_B['judge']['reasoning_effort'] == 'none', 'reasoning would reprice the batch line'
assert CFG_B['tag'] == TAG_B and CFG_B['judge']['provider'] == 'openai', CFG_B

# Both raters must read the same rubric bytes or Phi_A and Phi_B measure different things.
for path, cfg in ((JUDGE_CFG_A, CFG_A), (JUDGE_CFG_B, CFG_B)):
    assert cfg['template_file'] == 'prompts/judge_eval.txt', cfg['template_file']
RUBRIC = Path('prompts/judge_eval.txt')
RUBRIC_SHA = hashlib.sha256(RUBRIC.read_bytes()).hexdigest()
frozen = json.loads(Path('prompts/hashes.json').read_text(encoding='utf-8'))['templates']
assert RUBRIC_SHA == frozen['judge_eval.txt']['sha256'], 'the evaluation rubric has drifted'
print(f'rubric {RUBRIC_SHA[:16]} verified, read by both raters')

In [ ]:
# The measured val rates the pre-registration prices this pass at, upper end of each range.
RATE_A, RATE_B = 1.020e-3, 6.619e-4
N_CALLS = len(CONDS) * len(SEGMENTS)

PROJECTED = {'phi_a': RATE_A * N_CALLS, TAG_B: RATE_B * N_CALLS}
PROJECTED_TOTAL = sum(PROJECTED.values())
for name, rate in (('phi_a', RATE_A), (TAG_B, RATE_B)):
    print(f'{name:<8} {N_CALLS} calls at ${rate:.3e}  ${PROJECTED[name]:.2f}')
print(f'{"total":<8} {2 * N_CALLS} calls  ${PROJECTED_TOTAL:.2f}')

In [ ]:
SPEND_OK = False
AUTHORIZED_USD = 0.0

In [ ]:
assert SPEND_OK, 'set SPEND_OK = True to buy both raters'
assert AUTHORIZED_USD >= PROJECTED_TOTAL, (
    f'${AUTHORIZED_USD:.2f} authorized against a ${PROJECTED_TOTAL:.2f} projection'
)
print(subprocess.run(['git', 'log', '-1', '--format=%h %ad %s', '--date=short', '--',
                      'docs/budget.md'], capture_output=True, text=True).stdout)
print('the authorization this run spends against must already be a dated line in docs/budget.md')

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
%pip install -q anthropic==0.109.1 openai==2.41.1

In [ ]:
from src.eval.judge import judge_results_path, judge_segment_dir

USAGE_A = RESULTS / f'judge_{SPLIT}_usage.json'
USAGE_B = RESULTS / f'judge_{TAG_B}_{SPLIT}_usage.json'
JUDGE_A_PATH = judge_results_path(RESULTS, SPLIT)
JUDGE_B_PATH = judge_results_path(RESULTS, SPLIT, TAG_B)
SEG_A, SEG_B = judge_segment_dir(RESULTS, SPLIT), judge_segment_dir(RESULTS, SPLIT, TAG_B)


def prior_spend(path):
    """Cumulative (usd, calls) already on a rater's ledger for this split."""
    if not path.exists():
        return 0.0, 0
    cum = json.loads(path.read_text(encoding='utf-8'))['cumulative']
    return cum['cost_usd'], cum['calls']


PRIOR_A, PRIOR_B = prior_spend(USAGE_A), prior_spend(USAGE_B)
print(f'phi_a ledger ${PRIOR_A[0]:.4f} over {PRIOR_A[1]} calls')
print(f'{TAG_B} ledger ${PRIOR_B[0]:.4f} over {PRIOR_B[1]} calls')

In [ ]:
def reprice(usage_path, rate, label):
    """Realized per-call rate from a pilot's session block, against the projected rate."""
    u = json.loads(usage_path.read_text(encoding='utf-8'))
    assert u['priced'], f'{label}: unpriced model, so cost_usd is a floor rather than a bill'
    s = u['session']
    if s['calls'] == 0:
        # A re-run over a complete pilot cache buys nothing, so there is no new rate to read.
        print(f'{label}: 0 calls billed, its segments were already cached; '
              f'the projected ${rate:.3e} stands')
        return rate
    realized = s['cost_usd'] / s['calls']
    print(f"{label}: {s['calls']} calls, {s['prompt_tokens'] / s['calls']:.0f} in / "
          f"{s['completion_tokens'] / s['calls']:.0f} out per call, ${s['cost_usd']:.4f}")
    print(f'  ${realized:.3e}/call against a projected ${rate:.3e}  x{realized / rate:.2f}')
    assert realized <= 1.25 * rate, (
        f'{label}: ${realized:.3e}/call is more than 1.25x the projection; the full pass '
        f'would cost about ${realized * N_CALLS:.2f}'
    )
    return realized

In [ ]:
assert SPEND_OK
r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *CONDS, '--split', SPLIT,
                    '--config', JUDGE_CFG_A, '--limit', str(PILOT_N)], check=False)
assert r.returncode == 0, f'phi_a pilot exited {r.returncode}'
shutil.copy(USAGE_A, USAGE_A.with_name(f'judge_{SPLIT}_pilot_usage.json'))

In [ ]:
RATE_A_REAL = reprice(USAGE_A, RATE_A, 'phi_a pilot')
REVISED_A = RATE_A_REAL * N_CALLS
print(f'\nfull phi_a pass reprices to ${REVISED_A:.2f} against ${PROJECTED["phi_a"]:.2f} projected')
assert REVISED_A + PROJECTED[TAG_B] <= AUTHORIZED_USD, 'the realized rate exceeds the authorization'

In [ ]:
# Resumes over the pilot's cached segments, so only the remaining calls are bought.
r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *CONDS, '--split', SPLIT,
                    '--config', JUDGE_CFG_A], check=False)
assert r.returncode == 0, f'phi_a exited {r.returncode}'

In [ ]:
assert SPEND_OK
r = subprocess.run([PY, 'manage.py', 'judge_batch', '--conditions', *CONDS, '--split', SPLIT,
                    '--config', JUDGE_CFG_B, '--limit', str(PILOT_N), '--poll_interval', '15'],
                   check=False)
assert r.returncode == 0, f'{TAG_B} pilot exited {r.returncode}'
shutil.copy(USAGE_B, USAGE_B.with_name(f'judge_{TAG_B}_{SPLIT}_pilot_usage.json'))

In [ ]:
RATE_B_REAL = reprice(USAGE_B, RATE_B, f'{TAG_B} pilot')
REVISED_B = RATE_B_REAL * N_CALLS
print(f'\nfull {TAG_B} pass reprices to ${REVISED_B:.2f} against ${PROJECTED[TAG_B]:.2f} projected')
assert REVISED_A + REVISED_B <= AUTHORIZED_USD, 'the realized rates exceed the authorization'

In [ ]:
# The 24h completion window is the batch contract; a timeout here leaves the job running
# server-side and a re-run resumes polling it.
r = subprocess.run([PY, 'manage.py', 'judge_batch', '--conditions', *CONDS, '--split', SPLIT,
                    '--config', JUDGE_CFG_B], check=False)
assert r.returncode == 0, f'{TAG_B} exited {r.returncode}'

In [ ]:
PHI = {'phi_a': json.loads(JUDGE_A_PATH.read_text(encoding='utf-8')),
       TAG_B: json.loads(JUDGE_B_PATH.read_text(encoding='utf-8'))}
MODELS = {'phi_a': 'claude-haiku-4-5', TAG_B: 'gpt-5.6-terra'}
# This rater leaves a verdict unparsed on roughly 1-2% of segments; the primary judge does not.
COVERAGE_MIN = {'phi_a': 1.0, TAG_B: 0.97}

for rater, table in PHI.items():
    assert set(table) == set(CONDS), sorted(set(CONDS) - set(table))
    for cond in CONDS:
        rec = table[cond]
        assert rec['model'] == MODELS[rater], (rater, cond, rec['model'])
        assert rec['n'] == len(SEGMENTS), (rater, cond, rec['n'])
        assert rec['output_sha256'] == MANIFEST['rows'][cond]['output_sha256'], (
            f'{rater}/{cond}: scored bytes other than the ones the manifest records')
        assert rec['coverage'] >= COVERAGE_MIN[rater], (rater, cond, rec['coverage'])
        assert rec['template_sha256'] == PHI['phi_a']['zeroshot']['template_sha256']

print(f"{'condition':<18} {'Phi_A':>7} {'cov':>6} {'Phi_B':>7} {'cov':>6} {'B-A':>7}")
for cond in CONDS:
    a, b = PHI['phi_a'][cond], PHI[TAG_B][cond]
    print(f"{cond:<18} {a['mean']:7.3f} {a['coverage']:6.3f} {b['mean']:7.3f} "
          f"{b['coverage']:6.3f} {b['mean'] - a['mean']:+7.3f}")

In [ ]:
# The cache's own record of which generation bytes each rater scored, read back independently
# of the results files.
for rater, seg_dir in (('phi_a', SEG_A), (TAG_B, SEG_B)):
    meta = json.loads((seg_dir / '_meta.json').read_text(encoding='utf-8'))
    for cond in CONDS:
        assert meta['outputs'][cond]['sha256'] == MANIFEST['rows'][cond]['output_sha256'], \
            (rater, cond)
    print(f'{rater}: {len(meta["outputs"])} conditions bound in {seg_dir}/_meta.json')

In [ ]:
SPEND = {}
for rater, usage_path, prior in (('phi_a', USAGE_A, PRIOR_A), (TAG_B, USAGE_B, PRIOR_B)):
    u = json.loads(usage_path.read_text(encoding='utf-8'))
    assert u['priced'], f'{rater}: unpriced, so cost_usd is a floor rather than a bill'
    SPEND[rater] = {'model': u['model'],
                    'usd': round(u['cumulative']['cost_usd'] - prior[0], 4),
                    'calls': u['cumulative']['calls'] - prior[1],
                    'cumulative_usd': u['cumulative']['cost_usd']}
    print(f"{rater:<8} {u['model']:<18} ${SPEND[rater]['usd']:8.4f} over "
          f"{SPEND[rater]['calls']:6d} calls")

RATER_USD = round(sum(v['usd'] for v in SPEND.values()), 4)
assert RATER_USD <= AUTHORIZED_USD, f'${RATER_USD:.2f} spent against ${AUTHORIZED_USD:.2f}'
print(f'\nboth raters: ${RATER_USD:.2f} against ${AUTHORIZED_USD:.2f} authorized')

---
## 5 — Step 8: the ten confirmatory tests

Five contrasts on COMET and on `stylo_dist`, one Holm family, predicted signs fixed before the
pass. Nothing here is computed from a quantity that was not declared.

In [ ]:
from src.eval.metric_agreement import holm_bonferroni

CONF_BOOT = json.loads(CONF_BOOT_PATH.read_text(encoding='utf-8'))
assert (CONF_BOOT['n_resamples'], CONF_BOOT['seed'], CONF_BOOT['alpha']) == (N_BOOT, SEED, ALPHA)
# A dropped segment would put the two metrics on different evaluation sets.
assert all(rec['n'] == len(SEGMENTS) for rec in CONF_BOOT['comparisons']), CONF_BOOT_PATH

# (n, a, b, metric, predicted sign, val point estimate). Negative is better for stylo_dist.
FAMILY = [
    (1,  'peft',        'knn_fewshot', 'comet',      '+',    +0.0147),
    (2,  'peft',        'knn_fewshot', 'stylo_dist', '-',    -0.0753),
    (3,  'rlsf_w3_2.0', 'peft',        'comet',      '+',    +0.0021),
    (4,  'rlsf_w3_2.0', 'peft',        'stylo_dist', '-',    -0.0098),
    (5,  'afsp_full',   'knn_fewshot', 'comet',      '+',    +0.0014),
    (6,  'afsp_full',   'knn_fewshot', 'stylo_dist', '-',    -0.0622),
    (7,  'peft_afsp',   'peft',        'comet',      '+',    +0.0047),
    (8,  'peft_afsp',   'peft',        'stylo_dist', '+',    +0.0342),
    (9,  'sparse_knn',  'knn_fewshot', 'comet',      'none', +0.0007),
    (10, 'sparse_knn',  'knn_fewshot', 'stylo_dist', 'none', +0.0089),
]
assert len(FAMILY) == 10


def comet_contrast(a, b):
    for rec in CONF_BOOT['comparisons']:
        if (rec['a'], rec['b']) == (a, b):
            return {k: rec[k] for k in ('diff', 'ci_low', 'ci_high', 'p_value', 'significant')}
    raise KeyError(f'comet: {a} - {b} not in {CONF_BOOT_PATH}')


def stylo_contrast(a, b):
    """The ladder orders each pair better-rank-first, so an interval may need flipping."""
    for rec in STYLO['paired_all']:
        if {rec['a'], rec['b']} != {a, b}:
            continue
        flip = 1.0 if rec['a'] == a else -1.0
        lo, hi = sorted((flip * rec['ci_low'], flip * rec['ci_high']))
        return {'diff': flip * rec['diff'], 'ci_low': lo, 'ci_high': hi,
                'p_value': rec['p_value'], 'significant': rec['significant']}
    raise KeyError(f'stylo_dist: {a} - {b} not in {LADDER_PATH}')

In [ ]:
TESTS = []
for n, a, b, metric, sign, val in FAMILY:
    rec = comet_contrast(a, b) if metric == 'comet' else stylo_contrast(a, b)
    TESTS.append({'n': n, 'a': a, 'b': b, 'metric': metric, 'predicted_sign': sign,
                  'val': val, **rec})

P = [t['p_value'] for t in TESTS]
REJECT = holm_bonferroni(P, alpha=ALPHA)
order = sorted(range(len(P)), key=lambda i: P[i])
for rank, i in enumerate(order):
    TESTS[i]['holm_threshold'] = ALPHA / (len(P) - rank)
    TESTS[i]['holm_reject'] = bool(REJECT[i])

for t in TESTS:
    observed = '+' if t['diff'] > 0 else '-'
    if t['predicted_sign'] == 'none':
        t['verdict'] = 'discrepancy' if t['holm_reject'] else 'not detected'
    elif not t['holm_reject']:
        t['verdict'] = 'not replicated'
    else:
        t['verdict'] = 'replicated' if observed == t['predicted_sign'] else 'reversed'
    t['observed_sign'] = observed

In [ ]:
print(f"{'#':>2} {'contrast':<30} {'metric':<11} {'pred':>4} {'val':>9} {'test':>9} "
      f"{'95% CI':>21} {'p':>8} {'holm':>7} verdict")
for t in TESTS:
    ci = f"[{t['ci_low']:+.4f}, {t['ci_high']:+.4f}]"
    print(f"{t['n']:>2} {t['a'] + ' - ' + t['b']:<30} {t['metric']:<11} {t['predicted_sign']:>4} "
          f"{t['val']:+9.4f} {t['diff']:+9.4f} {ci:>21} {t['p_value']:8.4f} "
          f"{t['holm_threshold']:7.4f} {t['verdict']}")

replicated = [t['n'] for t in TESTS if t['verdict'] == 'replicated']
reversed_ = [t['n'] for t in TESTS if t['verdict'] == 'reversed']
discrepant = [t['n'] for t in TESTS if t['verdict'] == 'discrepancy']
print(f'\n{len(replicated)}/10 replicated: {replicated or "none"}')
print(f'predictions rejected in the opposite direction: {reversed_ or "none"}')
print(f'resolved on test where val had none: {discrepant or "none"}')
print('a test that does not reject is a failure to detect, not evidence of no effect')

In [ ]:
CONFIRMATORY_PATH = RESULTS / f'confirmatory_{SPLIT}.json'
CONFIRMATORY = {
    'split': SPLIT,
    'commit': HEAD,
    'preregistration': 'docs/preregistration_test.md',
    'correction': {'method': 'holm_bonferroni', 'alpha': ALPHA, 'family_size': len(TESTS)},
    'bootstrap': {'n_resamples': N_BOOT, 'seed': SEED, 'paired': True,
                  'n_segments': len(SEGMENTS)},
    'sources': {'comet': str(CONF_BOOT_PATH), 'stylo_dist': str(LADDER_PATH)},
    'output_sha256': {c: MANIFEST['rows'][c]['output_sha256'] for c in STUDY},
    'tests': TESTS,
    'summary': {'replicated': replicated, 'reversed': reversed_, 'discrepancy': discrepant},
}
CONFIRMATORY_PATH.write_text(json.dumps(CONFIRMATORY, indent=2) + '\n', encoding='utf-8')
print(f'Wrote {CONFIRMATORY_PATH}')

---
## 6 — Step 9: everything the family excludes

Reported, uncorrected as part of the confirmatory family, and load-bearing for no claim.

In [ ]:
# Both rater ladders, on the stored segment scores, so they cost nothing.
for tag in (None, TAG_B):
    subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', 'judge', '--split', SPLIT,
                    *(('--judge_tag', tag) if tag else ()),
                    '--conditions', *CONDS, '--baseline', 'zeroshot', '--adjacent',
                    '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                    '--out'], check=True)

In [ ]:
for tag in (None, TAG_B):
    cmd = [PY, 'manage.py', 'judge_ci', '--split', SPLIT, '--conditions', *CONDS,
           '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED)]
    if tag:
        cmd += ['--tag', tag]
    subprocess.run(cmd, check=True)

In [ ]:
subprocess.run([PY, 'manage.py', 'judge_agreement', '--split', SPLIT, '--conditions', *STUDY,
                '--tag_b', TAG_B, '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA),
                '--seed', str(SEED)], check=True)

In [ ]:
AGREEMENT = json.loads(
    (RESULTS / f'judge_agreement_{TAG_B}_{SPLIT}.json').read_text(encoding='utf-8'))
pooled = AGREEMENT['rater_agreement']['study_only']['pooled']
print(f"pooled n={pooled['n']}  qwk={pooled['qwk']['kappa']:+.3f}  "
      f"rho={pooled['spearman']['rho']:+.3f}  exact={pooled['exact_agreement']:.1%}")
print(f"severity offset A-B = {pooled['offset']['diff']:+.3f} "
      f"[{pooled['offset']['ci_low']:+.3f}, {pooled['offset']['ci_high']:+.3f}]")
flipped = [k for k, v in AGREEMENT['contrast_replication']['contrasts'].items()
           if not v['same_sign']]
print(f"identical system ranking: "
      f"{AGREEMENT['condition_ordering']['study_only']['identical_ranking']}")
print(f'sign flips between raters: {flipped or "none"}')

In [ ]:
# RQ4 over the twelve study conditions and both raters. The CLI's --raters map points at the
# val segment directories, so build() is called with this split's directories instead.
from src.eval.metric_agreement import build as metric_agreement_build

AGREE_PATH = RESULTS / f'metric_agreement_{SPLIT}.json'
RQ4 = metric_agreement_build(
    OUT, SPLIT,
    conditions=STUDY,
    reference='commercial_haiku',
    config_path=Path('configs/afsp_sweep.yaml'),
    rater_dirs={'phi_a': SEG_A, 'phi_b': SEG_B},
    comet_path=COMET_PATH,
    n_resamples=N_BOOT,
    alpha=ALPHA,
    seed=SEED,
)
AGREE_PATH.write_text(json.dumps(RQ4, indent=2) + '\n', encoding='utf-8')
print(f'Wrote {AGREE_PATH}')

In [ ]:
from src.eval.metric_agreement import _print_summary

_print_summary(RQ4)

In [ ]:
# The held-out prediction the pre-registration records and excludes: fixed in advance,
# carrying no weight, reported so it cannot be smuggled into the family afterwards.
DECOMP = json.loads((RESULTS / f'heldout_decomp_{SPLIT}.json').read_text(encoding='utf-8'))
assert DECOMP['reference'] == 'peft', DECOMP['reference']

print(f"peft dist_heldout {DECOMP['cells']['peft']['dist_heldout']:.4f}")
for cond in ('rlsf_w3_0.0', 'rlsf_w3_2.0', 'rlsf_w3_6.0'):
    cell = DECOMP['cells'][cond]
    d = cell['dist_heldout_delta']
    predicted = ' predicted negative' if cond in ('rlsf_w3_2.0', 'rlsf_w3_6.0') else ''
    print(f"{cond:<14} {cell['dist_heldout']:.4f}  delta {d['delta']:+.4f} "
          f"[{d['ci_low']:+.4f}, {d['ci_high']:+.4f}]  p={d['p_value']:.4f}{predicted}")

In [ ]:
# The two external rows, beside the study row each one is a diagnostic for. They change the
# generator family and support no study claim.
COMPANION = {'commercial_haiku': 'zeroshot', 'gpt56_sparse_knn': 'sparse_knn'}
print(f"{'row':<18} {'COMET':>8} {'stylo':>8} {'chrF':>7} {'BLEU':>7} {'Phi_A':>7} {'Phi_B':>7}")
for external, local in COMPANION.items():
    for cond in (external, local):
        print(f"{cond:<18} {COMET[cond]['system']:8.4f} "
              f"{STYLO['cells'][cond]['stylo_dist']:8.4f} {SURFACE[cond]['chrF']:7.2f} "
              f"{SURFACE[cond]['BLEU']:7.2f} {PHI['phi_a'][cond]['mean']:7.3f} "
              f"{PHI[TAG_B][cond]['mean']:7.3f}")
    print()

---
## 7 — Seal

In [ ]:
EXPLORATORY = [RESULTS / f'bootstrap_judge_{SPLIT}.json',
               RESULTS / f'bootstrap_judge_{TAG_B}_{SPLIT}.json',
               RESULTS / f'judge_ci_{SPLIT}.json',
               RESULTS / f'judge_ci_{TAG_B}_{SPLIT}.json',
               RESULTS / f'judge_agreement_{TAG_B}_{SPLIT}.json',
               AGREE_PATH]
WRITTEN = [*LOCAL_ARTIFACTS, JUDGE_A_PATH, JUDGE_B_PATH, USAGE_A, USAGE_B,
           CONFIRMATORY_PATH, *EXPLORATORY]

for p in WRITTEN:
    assert p.exists(), p
    print(f'{str(p):<52} {p.stat().st_size / 1024:8.1f} KiB')

# Nothing scored here may have moved the generations, the split, or any val artifact.
for cond in CONDS:
    path = OUT / f'{cond}_{SPLIT}.jsonl'
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST['rows'][cond]['output_sha256']
assert hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest() == TEST_SHA, 'test.jsonl changed'
dirty = subprocess.run(['git', 'status', '--porcelain', 'configs', 'outputs', 'results', 'data',
                        'docs/figures'], capture_output=True, text=True).stdout.splitlines()
unexpected = [line for line in dirty if 'test' not in line]
assert not unexpected, unexpected
print(f'\n{len(CONDS)} rows scored, ${RATER_USD:.2f} spent, no val artifact touched')

In [ ]:
import platform

import numpy
import sacrebleu

SCORING_MANIFEST = {
    'split': SPLIT,
    'commit': HEAD,
    'generation_manifest': {'path': str(OUT / 'test_manifest.json'),
                            'commit': MANIFEST['commit'],
                            'sha256': hashlib.sha256(
                                (OUT / 'test_manifest.json').read_bytes()).hexdigest()},
    'preregistration': 'docs/preregistration_test.md',
    'conditions': CONDS,
    'estimators': {'n_resamples': N_BOOT, 'alpha': ALPHA, 'seed': SEED, 'paired': True},
    'centroid': STYLO['centroid'],
    'raters': {'phi_a': {'model': MODELS['phi_a'], 'config': JUDGE_CFG_A, 'tag': None},
               'phi_b': {'model': MODELS[TAG_B], 'config': JUDGE_CFG_B, 'tag': TAG_B},
               'rubric_sha256': RUBRIC_SHA},
    'artifacts': {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in WRITTEN},
    'spend': {'authorized_usd': AUTHORIZED_USD, 'projected_usd': round(PROJECTED_TOTAL, 4),
              'actual_usd': RATER_USD, 'per_rater': SPEND},
    'confirmatory': CONFIRMATORY['summary'],
    'versions': {'python': platform.python_version(), 'numpy': numpy.__version__,
                 'sacrebleu': sacrebleu.__version__, 'comet_model': COMET['zeroshot']['model']},
}
SCORING_PATH = RESULTS / f'scoring_manifest_{SPLIT}.json'
SCORING_PATH.write_text(json.dumps(SCORING_MANIFEST, indent=2) + '\n', encoding='utf-8')
print(json.dumps({k: v for k, v in SCORING_MANIFEST.items() if k != 'artifacts'}, indent=2))

In [ ]:
!tar -czf test_scoring.tar.gz results/*_test.json results/*_test_usage.json \
    results/*_test_pilot_usage.json results/judge_test_segments results/judge_gpt_test_segments \
    docs/figures/heldout_decomp_by_omega_test.png
!ls -la test_scoring.tar.gz
!git status --short results docs/figures